# 🎨 TCS34725 센서 데이터 기반 MLP 색깔 분류

**목표**: 실제 RGB 센서(TCS34725)로 측정한 데이터를 사용하여 색깔 분류 MLP 모델 학습

**데이터**: 색깔 6종 × 거리 4종 × 50회 측정 = 1200개 샘플

| 거리  | 샘플 수 | 비고         |
|-------|---------|--------------|
| 0.5cm | 50개    | 근거리 측정  |
| 1cm   | 50개    | 중거리 측정  |
| 2cm   | 50개    | 원거리 측정  |
| 5cm   | 50개    | 먼거리 측정  |

**색깔**: 6종 (라벨당 200개)
- 🔴 Red (빨강): 200개
- 🟠 Orange (주황): 200개
- 🟡 Yellow (노랑): 200개
- 🟢 Green (초록): 200개
- 🔵 Blue (파랑): 200개
- 🟣 Purple (보라): 200개

**모델**: PyTorch MLP

In [17]:
# 필요한 라이브러리 import
import numpy as np
import pandas as pd
import serial
import serial.tools.list_ports
import time
import os
from datetime import datetime

In [18]:
# 🔌 사용 가능한 시리얼 포트 확인
print("사용 가능한 시리얼 포트:")
for port in serial.tools.list_ports.comports():
    print(f"  • {port.device}: {port.description}")

사용 가능한 시리얼 포트:
  • COM3: USB-SERIAL CH340(COM3)


In [22]:
# ⚙️ 설정
SERIAL_PORT = "COM3"  
BAUD_RATE = 9600

# 거리 설정
DISTANCES = ['0.5cm', '1cm', '2cm', '5cm']
SAMPLES_PER_DISTANCE = 50  # 거리당 샘플 수

# 색깔 설정
COLOR_NAMES = ['Red', 'Orange', 'Yellow', 'Green', 'Blue', 'Purple']
COLOR_KOREAN = ['빨강', '주황', '노랑', '초록', '파랑', '보라']
COLOR_EMOJI = ['🔴', '🟠', '🟡', '🟢', '🔵', '🟣']

# 총 샘플 계산
samples_per_color = SAMPLES_PER_DISTANCE * len(DISTANCES)  # 200개
total_samples = samples_per_color * len(COLOR_NAMES)  # 1200개

print(f"{'='*60}")
print(f"📊 데이터 수집 설정")
print(f"{'='*60}")
print(f"측정 거리: {', '.join(DISTANCES)}")
print(f"거리당 샘플: {SAMPLES_PER_DISTANCE}개")
print(f"색깔당 총 샘플: {samples_per_color}개 (= {SAMPLES_PER_DISTANCE} × {len(DISTANCES)}거리)")
print(f"전체 샘플 수: {total_samples}개 (= {samples_per_color} × {len(COLOR_NAMES)}색깔)")
print()
print("수집 대상 색깔:")
for i, (name, kr, emoji) in enumerate(zip(COLOR_NAMES, COLOR_KOREAN, COLOR_EMOJI)):
    print(f"  {i+1}. {emoji} {name} ({kr}) - {samples_per_color}개")

📊 데이터 수집 설정
측정 거리: 0.5cm, 1cm, 2cm, 5cm
거리당 샘플: 50개
색깔당 총 샘플: 200개 (= 50 × 4거리)
전체 샘플 수: 1200개 (= 200 × 6색깔)

수집 대상 색깔:
  1. 🔴 Red (빨강) - 200개
  2. 🟠 Orange (주황) - 200개
  3. 🟡 Yellow (노랑) - 200개
  4. 🟢 Green (초록) - 200개
  5. 🔵 Blue (파랑) - 200개
  6. 🟣 Purple (보라) - 200개


In [24]:
# 데이터 저장 리스트
collected_data = []
import re

def parse_rgb_data(line):
    """
    다양한 형식의 RGB 데이터를 파싱
    
    지원 형식:
    - "R : 3 G : 2 B : 1"
    - "R: 3 G: 2 B: 1"
    - "145,58,52"
    """
    # 형식 1: "R : 값 G : 값 B : 값" 또는 "R: 값 G: 값 B: 값"
    pattern = r'R\s*:\s*(\d+)\s*G\s*:\s*(\d+)\s*B\s*:\s*(\d+)'
    match = re.search(pattern, line, re.IGNORECASE)
    if match:
        r, g, b = int(match.group(1)), int(match.group(2)), int(match.group(3))
        return r, g, b
    
    # 형식 2: "값,값,값"
    if ',' in line:
        parts = line.split(',')
        if len(parts) >= 3:
            try:
                r, g, b = float(parts[0]), float(parts[1]), float(parts[2])
                return int(r), int(g), int(b)
            except:
                pass
    
    return None

def collect_color_data(color_idx, distance):
    """
    특정 색깔과 거리에서 RGB 데이터를 수집하는 함수
    
    Args:
        color_idx: 색깔 인덱스 (0~5)
        distance: 측정 거리 ('0.5cm', '1cm', '2cm')
    
    Returns:
        list: 수집된 RGB 데이터 리스트
    """
    color_name = COLOR_NAMES[color_idx]
    color_kr = COLOR_KOREAN[color_idx]
    color_emoji = COLOR_EMOJI[color_idx]
    
    data_list = []
    
    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=2)
        print(f"\n{'='*60}")
        print(f"{color_emoji} {color_name} ({color_kr}) - 거리: {distance}")
        print(f"{'='*60}")
        print(f"✓ 연결됨: {SERIAL_PORT} @ {BAUD_RATE} baud")
        print(f"\n📏 측정 거리: {distance}")
        print(f"🎯 목표: {SAMPLES_PER_DISTANCE}개 샘플 수집")
        print(f"📌 [엔터] 측정 | [q + 엔터] 종료")
        print(f"{'-'*60}")
        
        time.sleep(2)  # 아두이노 안정화 대기
        ser.reset_input_buffer()
        count = 0
        
        while count < SAMPLES_PER_DISTANCE:
            remaining = SAMPLES_PER_DISTANCE - count
            user_input = input(f"\n[{count+1}/{SAMPLES_PER_DISTANCE}] 엔터 = 측정, q = 종료 > ").strip().lower()
            
            if user_input == 'q':
                print("\n⏹️ 수집 중단")
                break
            
            # 버퍼 비우기
            ser.reset_input_buffer()
            time.sleep(0.1)
            
            print("📡 측정 중...", end=" ", flush=True)
            
            # 여러 번 시도
            measured = False
            for attempt in range(10):  # 최대 10번 시도
                if ser.in_waiting > 0:
                    try:
                        line = ser.readline().decode('utf-8', errors='ignore').strip()
                        
                        if line:
                            result = parse_rgb_data(line)
                            if result:
                                r, g, b = result
                                if 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
                                    count += 1
                                    data_list.append({
                                        'R': r,
                                        'G': g,
                                        'B': b,
                                        'label': color_name,
                                        'distance': distance
                                    })
                                    print(f"\n✅ RGB({r:3d}, {g:3d}, {b:3d}) @ {distance} - 샘플 #{count}")
                                    measured = True
                                    break
                    except Exception as e:
                        print(f"[오류: {e}]", end=" ")
                
                time.sleep(0.2)  # 200ms 대기
            
            if not measured:
                print("\n⚠️ 데이터 수신 실패. 다시 시도하세요.")
        
        print(f"\n{'-'*60}")
        print(f"✅ {color_emoji} {color_name} @ {distance} 수집 완료: {count}개")
        
    except serial.SerialException as e:
        print(f"❌ 연결 실패: {e}")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
            print("🔌 시리얼 포트 닫힘")
    
    return data_list

---
## 📥 데이터 수집 실행

아래 셀을 실행하여 데이터를 수집합니다.  
각 색깔별로 50개씩, 총 300개의 RGB 데이터를 수집합니다.

In [62]:
# 🎨 색깔 및 거리 선택 후 수집 실행
print("="*60)
print("🎨 수집할 색깔을 선택하세요")
print("="*60)
for i, (name, kr, emoji) in enumerate(zip(COLOR_NAMES, COLOR_KOREAN, COLOR_EMOJI)):
    print(f"  {i+1}. {emoji} {name} ({kr})")
print("  0.  취소")
print("="*60)

try:
    choice = int(input("\n색깔 번호 입력 (0=취소, 1~6): "))
    if choice == 0:
        print("\n취소되었습니다.")
    elif 1 <= choice <= 6:
        color_idx = choice - 1
        emoji = COLOR_EMOJI[color_idx]
        name = COLOR_NAMES[color_idx]
        kr = COLOR_KOREAN[color_idx]
        
        print(f"\n✓ 선택: {emoji} {name} ({kr})")
        
        # 거리 선택
        print(f"\n{'='*60}")
        print("📏 측정 거리를 선택하세요")
        print("="*60)
        for i, dist in enumerate(DISTANCES):
            print(f"  {i+1}. {dist}")
        print("  0.  취소")
        print("="*60)
        
        dist_choice = int(input("\n거리 번호 입력 (0=취소, 1~3): "))
        
        if dist_choice == 0:
            print("\n취소되었습니다.")
        elif 1 <= dist_choice <= len(DISTANCES):
            distance = DISTANCES[dist_choice - 1]
            
            print(f"\n{'='*60}")
            print(f"📋 수집 정보")
            print(f"{'='*60}")
            print(f"  색깔: {emoji} {name} ({kr})")
            print(f"  거리: {distance}")
            print(f"  샘플 수: {SAMPLES_PER_DISTANCE}개")
            print(f"{'='*60}")
            print(f"\n⚠️ 종료하려면 'q' 입력 후 엔터\n")
            
            # 데이터 수집
            collected_data = collect_color_data(color_idx, distance)
            
            # 수집 완료 후 자동 저장
            if collected_data:
                save_dir = "color_data"
                os.makedirs(save_dir, exist_ok=True)
                
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                # 파일명에 거리 정보 포함 (0.5cm -> 0_5cm)
                dist_str = distance.replace('.', '_')
                filename = f"{save_dir}/{name.lower()}_{dist_str}_rgb_data_{timestamp}.csv"
                
                df_single = pd.DataFrame(collected_data)
                df_single.to_csv(filename, index=False, encoding='utf-8')
                
                print(f"\n{'='*60}")
                print(f"💾 데이터 저장 완료!")
                print(f"  파일: {filename}")
                print(f"  수집된 샘플: {len(df_single)}개")
                print(f"  거리: {distance}")
                print(f"{'='*60}")
            else:
                print("\n⚠️ 수집된 데이터가 없습니다.")
        else:
            print("❌ 1~3 사이의 숫자를 입력하세요.")
    else:
        print("❌ 0~6 사이의 숫자를 입력하세요.")
except ValueError:
    print("❌ 숫자를 입력하세요.")
except KeyboardInterrupt:

    print("\n\n⚠️ 사용자가 중단했습니다. (Ctrl+C)")

🎨 수집할 색깔을 선택하세요
  1. 🔴 Red (빨강)
  2. 🟠 Orange (주황)
  3. 🟡 Yellow (노랑)
  4. 🟢 Green (초록)
  5. 🔵 Blue (파랑)
  6. 🟣 Purple (보라)
  0.  취소

취소되었습니다.


In [63]:
#  현재 수집된 데이터 확인 (거리별로 모두 보여주기)
if collected_data:
    df_current = pd.DataFrame(collected_data)
    print("="*50)
    print(" 현재 수집된 데이터 (거리별)")
    print("="*50)
    print(f"샘플 수: {len(df_current)}개")
    print(f"색깔: {df_current['label'].iloc[0]}")
    if 'distance' in df_current.columns:
        # 거리 값 정렬 및 표준화
        unique_dist = sorted(df_current['distance'].dropna().unique(), key=lambda x: float(x.replace('cm','')))
        print(f"수집 거리: {unique_dist}")
        print("\n거리별 샘플 수:")
        for dist in unique_dist:
            count = (df_current['distance'] == dist).sum()
            print(f"  {dist}: {count}개")
        print("\n거리별 RGB 통계:")
        for dist in unique_dist:
            sub = df_current[df_current['distance'] == dist]
            if len(sub) > 0:
                print(f"\n[{dist}] RGB 통계:")
                print(sub[['R','G','B']].describe().round(1))
    print(f"\n전체 RGB 통계:")
    print(df_current[['R', 'G', 'B']].describe().round(1))
    print(f"\n데이터 미리보기:")
    display(df_current)
else:
    print(" 수집된 데이터가 없습니다.")

 현재 수집된 데이터 (거리별)
샘플 수: 50개
색깔: Purple
수집 거리: ['5cm']

거리별 샘플 수:
  5cm: 50개

거리별 RGB 통계:

[5cm] RGB 통계:
           R     G     B
count   50.0  50.0  50.0
mean   152.2  52.5  49.0
std      3.8   2.3   1.6
min    145.0  48.0  45.0
25%    150.0  51.0  49.0
50%    153.0  52.0  49.0
75%    154.0  54.0  50.0
max    161.0  57.0  52.0

전체 RGB 통계:
           R     G     B
count   50.0  50.0  50.0
mean   152.2  52.5  49.0
std      3.8   2.3   1.6
min    145.0  48.0  45.0
25%    150.0  51.0  49.0
50%    153.0  52.0  49.0
75%    154.0  54.0  50.0
max    161.0  57.0  52.0

데이터 미리보기:


,R,G,B,label,distance
0,147,56,51,Purple,5cm
1,147,56,51,Purple,5cm
2,146,56,51,Purple,5cm
3,146,56,51,Purple,5cm
4,146,56,52,Purple,5cm
5,145,57,52,Purple,5cm
6,147,56,51,Purple,5cm
7,148,55,51,Purple,5cm
8,148,55,50,Purple,5cm
9,149,55,50,Purple,5cm


---
## 📂 전체 데이터 병합

모든 색깔의 CSV 파일을 하나로 합칩니다.

In [ ]:
# 📊 저장된 CSV 파일 목록 확인 (거리별 현황 포함)
save_dir = "color_data"
if os.path.exists(save_dir):
    files = [f for f in os.listdir(save_dir) if f.endswith('.csv') and not f.startswith('all_')]
    if files:
        print("="*70)
        print("📂 저장된 데이터 파일 목록")
        print("="*70)
        
        total_samples = 0
        # 색깔 × 거리별 카운트
        color_dist_counts = {name: {dist: 0 for dist in DISTANCES} for name in COLOR_NAMES}
        
        for i, f in enumerate(sorted(files)):
            filepath = f"{save_dir}/{f}"
            temp_df = pd.read_csv(filepath)
            size = os.path.getsize(filepath) / 1024
            
            label = temp_df['label'].iloc[0] if len(temp_df) > 0 else "Unknown"
            distance = temp_df['distance'].iloc[0] if 'distance' in temp_df.columns else "N/A"
            
            emoji = COLOR_EMOJI[COLOR_NAMES.index(label)] if label in COLOR_NAMES else "❓"
            print(f"  {i+1}. {emoji} {f}")
            print(f"      └─ {len(temp_df)}개, {distance}, {size:.1f}KB")
            
            total_samples += len(temp_df)
            if label in color_dist_counts and distance in DISTANCES:
                color_dist_counts[label][distance] += len(temp_df)
        
        print("="*70)
        print(f"📊 총 파일 수: {len(files)}개")
        print(f"📊 총 샘플 수: {total_samples}개")
        
        # 거리별 수집 현황 테이블
        print(f"\n{'='*70}")
        print("📏 색깔 × 거리별 수집 현황")
        print("="*70)
        
        # 헤더
        header = f"{'색깔':<12}" + "".join([f"{dist:>10}" for dist in DISTANCES]) + f"{'합계':>10}"
        print(header)
        print("-"*70)
        
        for name, kr, emoji in zip(COLOR_NAMES, COLOR_KOREAN, COLOR_EMOJI):
            row = f"{emoji} {name:<8}"
            total = 0
            for dist in DISTANCES:
                count = color_dist_counts[name][dist]
                status = "✅" if count >= SAMPLES_PER_DISTANCE else f"{count}"
                row += f"{status:>10}"
                total += count
            
            total_status = "✅" if total >= SAMPLES_PER_DISTANCE * len(DISTANCES) else f"{total}"
            row += f"{total_status:>10}"
            print(row)
        
        print("-"*70)
        # 거리별 합계
        dist_totals = f"{'합계':<12}"
        grand_total = 0
        for dist in DISTANCES:
            dist_sum = sum(color_dist_counts[name][dist] for name in COLOR_NAMES)
            dist_totals += f"{dist_sum:>10}"
            grand_total += dist_sum
        dist_totals += f"{grand_total:>10}"
        print(dist_totals)
        
    else:
        print("저장된 CSV 파일이 없습니다.")
else:
    print(f"'{save_dir}' 폴더가 없습니다.")

📂 저장된 데이터 파일 목록
  1. 🔵 blue_rgb_data_20260121_215233.csv (50개, 0.8KB)
  2. 🟢 green_rgb_data_20260121_215048.csv (50개, 0.8KB)
  3. 🟠 orange_rgb_data_20260121_214637.csv (50개, 0.9KB)
  4. 🟣 purple_rgb_data_20260121_215401.csv (50개, 0.9KB)
  5. 🔴 red_rgb_data_20260121_214253.csv (50개, 0.7KB)
  6. 🟡 yellow_rgb_data_20260121_214838.csv (50개, 0.9KB)
📊 총 파일 수: 6개
📊 총 샘플 수: 300개

색깔별 수집 현황:
  🔴 Red (빨강): 50/50개 ✅
  🟠 Orange (주황): 50/50개 ✅
  🟡 Yellow (노랑): 50/50개 ✅
  🟢 Green (초록): 50/50개 ✅
  🔵 Blue (파랑): 50/50개 ✅
  🟣 Purple (보라): 50/50개 ✅


In [74]:
# 🔗 모든 CSV 파일 병합 (거리 정보 포함) - 0.5cm, 1cm, 2cm만
save_dir = "color_data_distance"
valid_distances = ['0.5cm', '1cm', '2cm']  # 5cm 제외

if os.path.exists(save_dir):
    files = [f for f in os.listdir(save_dir) if f.endswith('.csv') and not f.startswith('all_')]
    
    # 유효한 거리의 파일만 필터링
    filtered_files = []
    for f in files:
        # 파일명에서 거리 추출 (예: blue_0_5cm_rgb_data_... → 0.5cm)
        if '_0_5cm_' in f:
            dist = '0.5cm'
        elif '_1cm_' in f:
            dist = '1cm'
        elif '_2cm_' in f:
            dist = '2cm'
        elif '_5cm_' in f:
            dist = '5cm'
        else:
            dist = None
        
        if dist in valid_distances:
            filtered_files.append(f)
    
    if filtered_files:
        all_data = []
        for f in filtered_files:
            temp_df = pd.read_csv(f"{save_dir}/{f}")
            all_data.append(temp_df)
        
        df_all = pd.concat(all_data, ignore_index=True)
        
        print("="*60)
        print("🔗 병합된 전체 데이터 (0.5cm, 1cm, 2cm만)")
        print("="*60)
        print(f"총 파일 수: {len(filtered_files)}개")
        print(f"총 샘플 수: {len(df_all)}개")
        print(f"포함된 거리: {valid_distances}")
        
        print(f"\n📊 클래스별 분포:")
        print(df_all['label'].value_counts())
        
        if 'distance' in df_all.columns:
            print(f"\n📏 거리별 분포:")
            print(df_all['distance'].value_counts())
            
            print(f"\n📊 색깔 × 거리별 분포:")
            print(pd.crosstab(df_all['label'], df_all['distance']))
        
        # 병합 파일 저장
        merged_filename = f"{save_dir}/all_color_05to2.csv"
        df_all.to_csv(merged_filename, index=False, encoding='utf-8')
        
        print(f"\n💾 병합 파일 저장: {merged_filename}")
        print(f"   컬럼: {list(df_all.columns)}")
        print(f"   샘플 수: {len(df_all)}개")
    else:
        print("병합할 CSV 파일이 없습니다 (유효한 거리 파일 없음).")
else:
    print(f"'{save_dir}' 폴더가 없습니다.")

🔗 병합된 전체 데이터 (0.5cm, 1cm, 2cm만)
총 파일 수: 18개
총 샘플 수: 900개
포함된 거리: ['0.5cm', '1cm', '2cm']

📊 클래스별 분포:
label
Blue      150
Green     150
Orange    150
Purple    150
Red       150
Yellow    150
Name: count, dtype: int64

📏 거리별 분포:
distance
0.5cm    300
1cm      300
2cm      300
Name: count, dtype: int64

📊 색깔 × 거리별 분포:
distance  0.5cm  1cm  2cm
label                    
Blue         50   50   50
Green        50   50   50
Orange       50   50   50
Purple       50   50   50
Red          50   50   50
Yellow       50   50   50

💾 병합 파일 저장: color_data_distance/all_color_05to2.csv
   컬럼: ['R', 'G', 'B', 'label', 'distance']
   샘플 수: 900개


---
## 📊 모든 수집된 데이터 출력

저장된 모든 CSV 파일의 데이터를 로드하여 색깔별로 거리별로 그룹화하여 출력합니다.


In [72]:
# 📊 모든 수집된 데이터 로드 및 출력
import glob

save_dir = "color_data_distance"
if os.path.exists(save_dir):
    # 모든 CSV 파일 찾기 (all_로 시작하는 병합 파일 제외)
    csv_files = glob.glob(f"{save_dir}/*.csv")
    csv_files = [f for f in csv_files if not os.path.basename(f).startswith('all_')]
    
    if csv_files:
        all_data = []
        for filepath in csv_files:
            try:
                temp_df = pd.read_csv(filepath)
                # distance 컬럼이 없으면 'N/A' 추가
                if 'distance' not in temp_df.columns:
                    temp_df['distance'] = 'N/A'
                all_data.append(temp_df)
            except Exception as e:
                print(f"❌ 파일 읽기 오류 {filepath}: {e}")
        
        if all_data:
            df_all = pd.concat(all_data, ignore_index=True)
            
            print("="*80)
            print("📊 모든 수집된 데이터 개요")
            print("="*80)
            print(f"총 파일 수: {len(csv_files)}개")
            print(f"총 샘플 수: {len(df_all)}개")
            print(f"컬럼: {list(df_all.columns)}")
            
            # 색깔별 거리별 분포
            if 'label' in df_all.columns and 'distance' in df_all.columns:
                print(f"\n{'='*80}")
                print("📏 색깔별 × 거리별 샘플 수")
                print("="*80)
                
                # 크로스탭으로 분포 표시 (거리 없는 것도 포함)
                dist_order = ['0.5cm', '1cm', '2cm', '5cm', 'N/A']
                available_dists = [d for d in dist_order if d in df_all['distance'].unique()]
                crosstab = pd.crosstab(df_all['label'], df_all['distance'], margins=True, margins_name='합계')
                # 거리 순서 정렬
                if all(d in crosstab.columns for d in available_dists):
                    crosstab = crosstab[available_dists + ['합계']]
                
                print(crosstab)
                
                # 각 색깔별로 거리별 데이터 출력
                print(f"\n{'='*80}")
                print("🎨 색깔별 데이터 상세")
                print("="*80)
                
                for color in COLOR_NAMES:
                    color_data = df_all[df_all['label'] == color]
                    if len(color_data) > 0:
                        emoji = COLOR_EMOJI[COLOR_NAMES.index(color)]
                        print(f"\n{emoji} {color} ({COLOR_KOREAN[COLOR_NAMES.index(color)]})")
                        print("-" * 60)
                        
                        # 거리별 통계
                        for dist in dist_order:
                            if dist in color_data['distance'].unique():
                                dist_data = color_data[color_data['distance'] == dist]
                                if len(dist_data) > 0:
                                    print(f"\n📏 거리: {dist} ({len(dist_data)}개 샘플)")
                                    print("RGB 통계:")
                                    print(dist_data[['R','G','B']].describe().round(1))
                                    print("데이터 미리보기:")
                                    display(dist_data.head())
                            else:
                                print(f"\n📏 거리: {dist} - 데이터 없음")
            else:
                print("❌ 'label' 또는 'distance' 컬럼이 없습니다.")
            
            # 전체 데이터 미리보기
            print(f"\n{'='*80}")
            print("📋 전체 데이터 미리보기 (처음 20개)")
            print("="*80)
            display(df_all.head(20))
            
            # 전체 RGB 통계
            print(f"\n{'='*80}")
            print("📊 전체 RGB 통계")
            print("="*80)
            print(df_all[['R','G','B']].describe().round(1))
            
        else:
            print("❌ 유효한 데이터 파일이 없습니다.")
    else:
        print("❌ CSV 파일이 없습니다.")
else:
    print(f"❌ '{save_dir}' 폴더가 없습니다.")

📊 모든 수집된 데이터 개요
총 파일 수: 24개
총 샘플 수: 1200개
컬럼: ['R', 'G', 'B', 'label', 'distance']

📏 색깔별 × 거리별 샘플 수
distance  0.5cm  1cm  2cm  5cm    합계
label                               
Blue         50   50   50   50   200
Green        50   50   50   50   200
Orange       50   50   50   50   200
Purple       50   50   50   50   200
Red          50   50   50   50   200
Yellow       50   50   50   50   200
합계          300  300  300  300  1200

🎨 색깔별 데이터 상세

🔴 Red (빨강)
------------------------------------------------------------

📏 거리: 0.5cm (50개 샘플)
RGB 통계:
           R     G     B
count   50.0  50.0  50.0
mean   170.5  47.9  35.2
std      1.1   0.7   0.7
min    168.0  44.0  33.0
25%    170.0  48.0  35.0
50%    170.0  48.0  35.0
75%    171.0  48.0  36.0
max    177.0  49.0  37.0
데이터 미리보기:


,R,G,B,label,distance
800,170,48,36,Red,0.5cm
801,171,47,35,Red,0.5cm
802,169,48,36,Red,0.5cm
803,170,48,35,Red,0.5cm
804,171,48,35,Red,0.5cm



📏 거리: 1cm (50개 샘플)
RGB 통계:
           R     G     B
count   50.0  50.0  50.0
mean   170.1  48.0  35.4
std      0.5   0.0   0.5
min    169.0  48.0  35.0
25%    170.0  48.0  35.0
50%    170.0  48.0  35.0
75%    170.0  48.0  36.0
max    171.0  48.0  37.0
데이터 미리보기:


,R,G,B,label,distance
850,170,48,35,Red,1cm
851,170,48,35,Red,1cm
852,170,48,35,Red,1cm
853,170,48,35,Red,1cm
854,170,48,35,Red,1cm



📏 거리: 2cm (50개 샘플)
RGB 통계:
           R     G     B
count   50.0  50.0  50.0
mean   170.5  47.3  36.0
std      0.5   0.5   0.2
min    169.0  47.0  35.0
25%    170.0  47.0  36.0
50%    171.0  47.0  36.0
75%    171.0  48.0  36.0
max    171.0  48.0  36.0
데이터 미리보기:


...


📏 거리: 5cm (50개 샘플)
RGB 통계:
           R     G     B
count   50.0  50.0  50.0
mean   152.2  52.5  49.0
std      3.8   2.3   1.6
min    145.0  48.0  45.0
25%    150.0  51.0  49.0
50%    153.0  52.0  49.0
75%    154.0  54.0  50.0
max    161.0  57.0  52.0
데이터 미리보기:


,R,G,B,label,distance
750,147,56,51,Purple,5cm
751,147,56,51,Purple,5cm
752,146,56,51,Purple,5cm
753,146,56,51,Purple,5cm
754,146,56,52,Purple,5cm



📏 거리: N/A - 데이터 없음

📋 전체 데이터 미리보기 (처음 20개)


,R,G,B,label,distance
0,77,84,92,Blue,0.5cm
1,75,85,93,Blue,0.5cm
2,75,85,93,Blue,0.5cm
3,74,86,94,Blue,0.5cm
4,74,86,94,Blue,0.5cm
5,75,86,93,Blue,0.5cm
6,74,86,94,Blue,0.5cm
7,74,86,94,Blue,0.5cm
8,75,86,93,Blue,0.5cm
9,75,85,93,Blue,0.5cm



📊 전체 RGB 통계
            R       G       B
count  1200.0  1200.0  1200.0
mean    134.4    68.7    50.4
std      30.0    17.0    18.3
min      72.0    44.0    32.0
25%     110.0    56.0    35.0
50%     137.0    63.0    42.5
75%     162.0    82.0    59.0
max     177.0   110.0    95.0
